In [2]:
# Name: Ahmad Hamizan Bin Mohd Ali
# Student ID: BIS01086549

# Import Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer 
from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel 
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Load the Data 
print("1. Loading dataset...")
df = pd.read_csv('news_dataset.csv', lineterminator='\n', on_bad_lines='skip')

# Use only the 'text' column and drop null values 
df = df[['text']].dropna()
documents = df['text'].tolist()

# Preprocess the Data
print("2. Preprocessing text (this may take a moment)...")
stop_words = set(stopwords.words('english')) 
lemmatizer = WordNetLemmatizer() 
stemmer = PorterStemmer() 

def preprocess_text(text):
    tokens = word_tokenize(str(text).lower()) 
    tokens = [token for token in tokens if token.isalnum()] 
    tokens = [token for token in tokens if token not in stop_words] 
    
    # Apply BOTH Stemming and Lemmatization 
    tokens = [stemmer.stem(token) for token in tokens] 
    tokens = [lemmatizer.lemmatize(token) for token in tokens] 
    
    return tokens 

preprocessed_documents = [preprocess_text(doc) for doc in documents] 

# Create document-term matrix
dictionary = corpora.Dictionary(preprocessed_documents)

# FIX: Lowered 'no_below' from 15 to 2 so we don't accidentally delete all our words!
dictionary.filter_extremes(no_below=2, no_above=0.5)

corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

# Run LDA
print("3. Training LDA Model (Please wait, this might take a few minutes)...")
lda_model = LdaModel(corpus, num_topics=4, id2word=dictionary, passes=15) 
print("-> Model training complete!")

# Evaluate the LDA model using Coherence score 
print("4. Calculating Coherence Score...")
coherence_model_lda = CoherenceModel(model=lda_model, texts=preprocessed_documents, dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f'\n--- MODEL EVALUATION ---')
print(f'Coherence Score: {coherence_lda:.4f}\n')

# Interpret Results
article_labels = []

for i, doc in enumerate(preprocessed_documents):
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    if topics: # Added a safety check here to prevent crashes if a document is completely empty
        dominant_topic = max(topics, key=lambda x: x[1])[0]
    else:
        dominant_topic = -1 
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

print("--- Table with Articles and Topic ---")
print(df_result.head(10)) 
print("\n")

# Print the top terms for each topic with weight
print("--- Top Terms for Each Topic ---")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()
    
print("Processing Finished!")

#Interpretation of Coherence Score and Topics:
#The LDA model generated a Coherence Score (C_v) of 0.5516. 
The coherence score measures how semantically related the words within a topic are to each other, which helps indicate how human interpretable the topics are. 
A score of 0.55 suggests the extracted topics capture a moderate to strong underlying structure in the news dataset. 
Looking at the top terms, the model has identified clear themes: Topic 0 appears focused on national security and politics ("trump", "state", "court", "law"), Topic 1 centers on health and medicine ("health", "care", "patient", "drug"), Topic 2 covers social/economic issues and education ("school", "student", "job", "money"), and Topic 3 relates to international relations and conflict ("country", "war", "russia", "military").

1. Loading dataset...
2. Preprocessing text (this may take a moment)...
3. Training LDA Model (Please wait, this might take a few minutes)...
-> Model training complete!
4. Calculating Coherence Score...

--- MODEL EVALUATION ---
Coherence Score: 0.6306

--- Table with Articles and Topic ---
                                             Article  Topic
0  I was wondering if anyone out there could enli...      0
1  I recently posted an article asking what kind ...      3
2  \nIt depends on your priorities.  A lot of peo...      0
3  an excellent automatic can be found in the sub...      0
4  : Ford and his automobile.  I need information...      0
5  \nYo! Watch the attributions--I didn't say tha...      0
6  \nYou can avoid these problems entirely by ins...      3
7  I have a 1986 Acura Integra 5 speed with 95,00...      3
8  \nassuming yours is a non turbo MR2, the gruff...      1
9  \n\nIn addition to restricted mileage, many cl...      1


--- Top Terms for Each Topic ---
Topic 0:
- "